<a href="https://colab.research.google.com/github/ga426553-sudo/Classification-of-Breast-Cancer-Subtypes-machine-learning---CuMiDa-22820/blob/main/Quasiconstant_SMOTE_TrainTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Primer código - Cáncer de Mama 🌸
**Preprocesamiento** 🔨

### Carga de archivos 🌺

In [ ]:
# ============================================
# CÓDIGO 1: PREPROCESAMIENTO DE DATOS (CORREGIDO)
# SIN DATA LEAKAGE
# ============================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pickle
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("🔧 CÓDIGO 1: PREPROCESAMIENTO (SIN DATA LEAKAGE)")
print("="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔧 CÓDIGO 1: PREPROCESAMIENTO (SIN DATA LEAKAGE)


### Analizar estructura del Dataset 🌺

In [ ]:
# 1. CARGAR DATOS
# =================
file_path = '/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/Breast_GSE22820.csv'  # AJUSTA ESTA RUTA
df = pd.read_csv(file_path, index_col=0)

print(f"\n📂 Dataset cargado: Breast_GSE22820")
print(f"Muestras totales: {df.shape[0]}")
print(f"Genes totales: {df.shape[1] - 1}")


📂 Dataset cargado: Breast_GSE22820
Muestras totales: 139
Genes totales: 33579


### Separación de variables X (característica) y Y (objetivo) 🌺

In [ ]:
# 2. SEPARAR X e y
# =================
class_column = 'type'  # VERIFICA QUE SEA EL NOMBRE CORRECTO
X = df.drop(columns=[class_column]).values
y = df[class_column].values
feature_names = df.drop(columns=[class_column]).columns.tolist()

print(f"\n📊 Distribución original:")
unique, counts = np.unique(y, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"   {cls}: {count}")


📊 Distribución original:
   normal: 10
   primary_breast_cancer: 129


### Etiquetas 🌺

In [ ]:
# 3. CODIFICAR ETIQUETAS
# =======================
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"\n🔤 Clases codificadas: 0={le.classes_[0]}, 1={le.classes_[1]}")


🔤 Clases codificadas: 0=normal, 1=primary_breast_cancer


### División en Train y Test 🌺

In [ ]:
# 4. PRIMERO: DIVIDIR EN TRAIN/TEST (¡LO MÁS IMPORTANTE!)
# =========================================================
print(f"\n--- PASO 1: Dividiendo en train/test (80/20) ---")
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train: {X_train.shape[0]} muestras ({np.sum(y_train==0)} normales, {np.sum(y_train==1)} tumores)")
print(f"Test: {X_test.shape[0]} muestras ({np.sum(y_test==0)} normales, {np.sum(y_test==1)} tumores)")


--- PASO 1: Dividiendo en train/test (80/20) ---
Train: 111 muestras (8 normales, 103 tumores)
Test: 28 muestras (2 normales, 26 tumores)


### Quasi-constant 🌺

In [ ]:
# 5. AHORA: VarianceThreshold SOLO EN TRAIN
# ===========================================
print(f"\n--- PASO 2: Aplicando filtro de varianza (threshold=0.01) SOLO EN TRAIN ---")

selector_var = VarianceThreshold(threshold=0.01)
X_train_var = selector_var.fit_transform(X_train)  # ✅ FIT solo en train
X_test_var = selector_var.transform(X_test)        # ✅ TRANSFORM en test

# Obtener los índices de las características seleccionadas
selected_features_mask = selector_var.get_support()
selected_feature_names = [feature_names[i] for i in range(len(feature_names)) if selected_features_mask[i]]

print(f"Genes seleccionados en train: {X_train_var.shape[1]}")
print(f"Genes eliminados: {len(feature_names) - X_train_var.shape[1]}")
print(f"Primeros 10 genes seleccionados: {selected_feature_names[:10]}")



--- PASO 2: Aplicando filtro de varianza (threshold=0.01) SOLO EN TRAIN ---
Genes seleccionados en train: 31215
Genes eliminados: 2364
Primeros 10 genes seleccionados: ['NM_004900', 'AA085955', 'NM_014616', 'AK092846', 'NM_001539', 'THC2450799', 'NM_006709', 'NM_000978', 'T12590', 'NM_001017']


### SMOTE 🌺

In [ ]:
# 6. AHORA: SMOTE SOLO EN TRAIN (después de VarianceThreshold)
# ==============================================================
print(f"\n--- PASO 3: Aplicando SMOTE SOLO EN TRAIN ---")
print(f"Antes de SMOTE (train): {pd.Series(y_train).value_counts().to_dict()}")

smote = SMOTE(sampling_strategy='auto', random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_var, y_train)  # ✅ SOLO train

print(f"Después de SMOTE (train): {pd.Series(y_train_resampled).value_counts().to_dict()}")
print(f"Test se mantiene intacto: {pd.Series(y_test).value_counts().to_dict()}")


--- PASO 3: Aplicando SMOTE SOLO EN TRAIN ---
Antes de SMOTE (train): {1: 103, 0: 8}
Después de SMOTE (train): {1: 103, 0: 103}
Test se mantiene intacto: {1: 26, 0: 2}


### Verificar que no hay data leakage 🌺

In [ ]:
# 7. VERIFICACIÓN FINAL
# ======================
print(f"\n--- VERIFICACIÓN: Sin data leakage ---")
print(f"✅ Train después de SMOTE: {X_train_resampled.shape}")
print(f"✅ Test original (sin tocar): {X_test_var.shape}")
print(f"   (Importante: test conserva su distribución original)")


--- VERIFICACIÓN: Sin data leakage ---
✅ Train después de SMOTE: (206, 31215)
✅ Test original (sin tocar): (28, 31215)
   (Importante: test conserva su distribución original)


### Guardar los archivos 🌺

In [ ]:
# 8. GUARDAR DATOS PROCESADOS
# =============================
print(f"\n--- Guardando datos procesados ---")

# Guardar datos de entrenamiento (balanceados)
np.save('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/X_train_balanced.npy', X_train_resampled)
np.save('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_train_balanced.npy', y_train_resampled)

# Guardar datos de prueba (originales, sin balancear)
np.save('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/X_test_clean.npy', X_test_var)
np.save('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_test_clean.npy', y_test)

# Guardar nombres de genes seleccionados
with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/feature_names_filtered.pkl', 'wb') as f:
    pickle.dump(selected_feature_names, f)

# Guardar label encoder para uso futuro
with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Guardar información del preprocesamiento
preproc_info = {
    'n_genes_original': len(feature_names),
    'n_genes_filtered': len(selected_feature_names),
    'train_samples_before_smote': X_train.shape[0],
    'train_samples_after_smote': X_train_resampled.shape[0],
    'test_samples': X_test_var.shape[0],
    'class_names': le.classes_.tolist(),
    'threshold_variance': 0.01,
    'smote_params': {'k_neighbors': 5, 'random_state': 42}
}

with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/preproc_info.pkl', 'wb') as f:
    pickle.dump(preproc_info, f)

print("\n✅ DATOS GUARDADOS EN GOOGLE DRIVE")
print("Archivos guardados:")
print("   - X_train_balanced.npy (train balanceado)")
print("   - y_train_balanced.npy")
print("   - X_test_clean.npy (test original)")
print("   - y_test_clean.npy")
print("   - feature_names_filtered.pkl")
print("   - label_encoder.pkl")
print("   - preproc_info.pkl")

print(f"\n📊 Resumen final:")
print(f"   Train (balanceado): {X_train_resampled.shape[0]} muestras, {X_train_resampled.shape[1]} genes")
print(f"   Test (original): {X_test_var.shape[0]} muestras, {X_test_var.shape[1]} genes")


--- Guardando datos procesados ---

✅ DATOS GUARDADOS EN GOOGLE DRIVE
Archivos guardados:
   - X_train_balanced.npy (train balanceado)
   - y_train_balanced.npy
   - X_test_clean.npy (test original)
   - y_test_clean.npy
   - feature_names_filtered.pkl
   - label_encoder.pkl
   - preproc_info.pkl

📊 Resumen final:
   Train (balanceado): 206 muestras, 31215 genes
   Test (original): 28 muestras, 31215 genes
